In [1]:
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
from torchvision.transforms import v2

import sys
import os
import matplotlib.pyplot as plt
import numpy as np

import json
from tqdm import tqdm
from PIL import Image

plt.style.use('dark_background')

# Сохранение и загрузка модели

In [ ]:
# Сохранение модели, тензоров и словарей
torch.save(obj, PATH)

# Загрузка модели, тензоров и словарей
torch.load(PATH)

In [2]:
dict_1 = {'key_1': torch.tensor([1, 2, 3]), 'key_2': torch.tensor([2, 3, 4])}
dict_1

{'key_1': tensor([1, 2, 3]), 'key_2': tensor([2, 3, 4])}

In [7]:
torch.save(dict_1, 'dict_1.pt')

In [8]:
new_dict = torch.load('dict_1.pt')
new_dict

{'key_1': tensor([1, 2, 3]), 'key_2': tensor([2, 3, 4])}

## Сохранение модели

In [20]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [21]:
class DataSetReg(Dataset):
    def __init__(self, path, transform=None):
        self.path = path
        self.transform = transform

        self.list_name_file = os.listdir(path)
        if 'coords.json' in self.list_name_file:
            self.list_name_file.remove('coords.json')

        self.len_dataset = len(self.list_name_file)

        with open(os.path.join(self.path, 'coords.json'), 'r') as fl:
            self.dict_coords = json.load(fl)

    def __len__(self):
        return self.len_dataset

    def __getitem__(self, index):
        name_file = self.list_name_file[index]
        path_img = os.path.join(self.path, name_file)

        img = np.array(Image.open(path_img))
        coord = np.array(self.dict_coords[name_file])

        if self.transform is not None:
            img = self.transform(img)

        return img, coord

In [22]:
# Преобразования для изображения
transform = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.5,), std=(0.5,))
    ]
)

In [23]:
dataset = DataSetReg('/drive/MyDrive/Colab_Notebooks/content/dataset/', transform=transform) if 'google.colab' in sys.modules \
    else DataSetReg('/Users/SashaCurry/PyCharmMiscProject/content/dataset/', transform=transform)

In [24]:
train_set, val_set, test_set = random_split(dataset, [0.7, 0.1, 0.2])

In [25]:
# Создание загрузчиков
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16, shuffle=False)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

In [26]:
class MyModel(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.layer_1 = nn.Linear(input, 128)
        self.layer_2 = nn.Linear(128, output)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation(x)
        out = self.layer_2(x)
        return out


model = MyModel(64*64, 2).to(device)

In [27]:
loss_model = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
lr_sheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt,
                                                         mode='min',
                                                         factor=0.1,
                                                         patience=5,
                                                         threshold=0.0001,
                                                         threshold_mode='rel',
                                                         cooldown=0,
                                                         min_lr=0,
                                                         eps=1e-08)

In [28]:
EPOCHS = 15
train_loss = []
train_acc = []
val_loss = []
val_acc = []
lr_list = []
best_loss = None
best_loss_epoch = None

In [11]:
# Цикл обучения
for epoch in range(EPOCHS):
    # Тренировка модели
    running_train_loss = []
    mean_train_loss = 0
    train_loop = tqdm(train_loader, leave=False) # создание progress bar
    true_answer = 0

    model.train()
    for x, targets in train_loop:
        # Данные
        # (batch_size, 1, 64, 64) -> (batch_size, 64*64)
        x = x.reshape(-1, 64*64).to(device)
        targets = targets.to(torch.float32).to(device)

        # Прямой проход + расчет ошибки модели
        pred = model(x)
        loss = loss_model(pred, targets)

        # Обратный проход
        opt.zero_grad()
        loss.backward()
        # Шаг оптимизации
        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss) / len(running_train_loss)

        true_answer += (torch.round(pred) == targets).all(dim=1).sum().item()

        train_loop.set_description(f'Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}')

    # Расчет значения метрики
    running_train_acc = true_answer / len(train_set)

    # Сохранение значения функции потерь и метрики
    train_loss.append(mean_train_loss)
    train_acc.append(running_train_acc)

    # Проверка модели (валидация)
    model.eval()
    with torch.no_grad():
        running_val_loss = []
        true_answer = 0
        for x, targets in val_loader:
            # Данные
            # (batch_size, 1, 64, 64) -> (batch_size, 64*64)
            x = x.reshape(-1, 64*64).to(device)
            targets = targets.to(torch.float32).to(device)

            # Прямой проход + расчет ошибки модели
            pred = model(x)
            loss = loss_model(pred, targets)

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss) / len(running_val_loss)

            true_answer += (torch.round(pred) == targets).all(dim=1).sum().item()

        # Расчет значения метрики
        running_val_acc = true_answer / len(val_set)

        # Сохранение значения функции потерь и метрики
        val_loss.append(mean_val_loss)
        val_acc.append(running_val_acc)

        lr_sheduler.step(mean_val_loss)
        lr = opt.param_groups[0]['lr']
        lr_list.append(lr)

        print(f'Epoch [{epoch+1}/{EPOCHS}]: train_loss={mean_train_loss:.4f}, train_acc={running_train_acc:.4f}, val_loss={mean_val_loss:.4f}, val_acc={running_val_acc:.4f}, lr={lr:.4f}')

        if best_loss is None:
            best_loss = mean_val_loss
            best_loss_epoch = epoch + 1

        if mean_val_loss < best_loss - best_loss*0.05:
            if os.path.exists(f'model_state_dict_epoch_{best_loss_epoch}.pt'):
                os.remove(f'model_state_dict_epoch_{best_loss_epoch}.pt')

            best_loss = mean_val_loss
            best_loss_epoch = epoch + 1

            torch.save(model.state_dict(), f'model_state_dict_epoch_{best_loss_epoch}.pt')
            print(f'На эпохе {epoch+1} сохранена модель со значением loss на валидационной выборе = {mean_val_loss:.4f}', end='\n\n')


Epoch [1/15]: train_loss=1.6229, train_acc=0.6309, val_loss=0.1011, val_acc=0.7896, lr=0.0010


Epoch [2/15]: train_loss=0.1395, train_acc=0.6780, val_loss=0.0830, val_acc=0.8427, lr=0.0010
На эпохе 2 сохранена модель со значением loss на валидационной выборе = 0.0830



Epoch [3/15]: train_loss=0.1118, train_acc=0.7631, val_loss=0.0812, val_acc=0.8472, lr=0.0010


Epoch [4/15]: train_loss=0.0950, train_acc=0.8134, val_loss=0.0648, val_acc=0.9038, lr=0.0010
На эпохе 4 сохранена модель со значением loss на валидационной выборе = 0.0648



Epoch [5/15]: train_loss=0.0837, train_acc=0.8478, val_loss=0.0837, val_acc=0.8601, lr=0.0010


...

Epoch [11/15]: train_loss=0.0628, train_acc=0.9127, val_loss=0.1255, val_acc=0.6748, lr=0.0010


Epoch [12/15]: train_loss=0.0589, train_acc=0.9237, val_loss=0.0438, val_acc=0.9656, lr=0.0010
На эпохе 12 сохранена модель со значением loss на валидационной выборе = 0.0438



Epoch [13/15]: train_loss=0.0572, train_acc=0.9287, val_loss=0.0799, val_acc=0.8764, lr=0.0010


Epoch [14/15]: train_loss=0.0549, train_acc=0.9344, val_loss=0.0414, val_acc=0.9695, lr=0.0010
На эпохе 14 сохранена модель со значением loss на валидационной выборе = 0.0414



Epoch [15/15]: train_loss=0.0539, train_acc=0.9367, val_loss=0.0550, val_acc=0.9392, lr=0.0010


## Загрузка модели

In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [14]:
# Загрузка параметров модели
param_model = torch.load('model_state_dict_epoch_14.pt', map_location=device)
param_model

OrderedDict([('layer_1.weight',
              tensor([[ 0.0018,  0.0111,  0.0123,  ...,  0.0149,  0.0066, -0.0089],
                      [-0.0010, -0.0092,  0.0204,  ...,  0.0185, -0.0058,  0.0133],
                      [-0.0208, -0.0067,  0.0228,  ..., -0.0084, -0.0261, -0.0028],
                      ...,
                      [-0.0014,  0.0109,  0.0105,  ...,  0.0042,  0.0082,  0.0095],
                      [ 0.0111, -0.0003,  0.0031,  ..., -0.0033,  0.0021,  0.0116],
                      [ 0.0113,  0.0209, -0.0080,  ..., -0.0013,  0.0009,  0.0209]],
                     device='cuda:0')),
             ('layer_1.bias',
              tensor([ 1.2311e-02, -1.8213e-02,  6.3386e-02, -7.8961e-03,  2.9502e-02,
                       6.7848e-03,  1.7959e-03, -2.5152e-03,  2.0618e-02,  2.8279e-03,
                      -2.7101e-03, -1.6291e-02,  1.9527e-03, -1.5430e-03, -9.5623e-03,
                       8.6391e-02, -6.7305e-03,  5.3980e-04, -8.6304e-03, -1.5434e-02,
                  

In [15]:
# Создание новой модели
new_model = MyModel(64*64, 2).to(device)

In [16]:
# Меняем параметры модели
new_model.load_state_dict(param_model)

<All keys matched successfully>

In [17]:
new_model.eval()
with torch.no_grad():
    running_val_loss = []
    true_answer = 0
    for x, targets in val_loader:
        # Данные
        # (batch_size, 1, 64, 64) -> (batch_size, 64*64)
        x = x.reshape(-1, 64*64).to(device)
        targets = targets.to(device)

        # Прямой проход + расчет ошибки модели
        pred = new_model(x)
        loss = loss_model(pred, targets)

        running_val_loss.append(loss.item())
        mean_val_loss = sum(running_val_loss)/len(running_val_loss)

        true_answer += (torch.round(pred) == targets).all(dim=1).sum().item()

# Расчет значения метрики
running_val_acc = true_answer / len(val_set)

print(f'val_loss = {mean_val_loss:.4f}, val_acc = {running_val_acc:.4f}')

val_loss = 0.0414, val_acc = 0.9695


## Сохранение модели + шедулера + метрик + эпох + другая информация

In [29]:
str_info = '''
class MyModel(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.layer_1 = nn.Linear(input, 128)
        self.layer_2 = nn.Linear(128, output)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.act(x)
        out = self.layer_2(x)
        return out

new_model = MyModel(64*64, 2)
'''

EPOCHS = 30
save_epoch = 14
best_loss = None

In [31]:
checkpoint = {
    'info': str_info,
    'state_model': model.state_dict(),
    'state_opt': opt.state_dict(),
    'state_lr_scheduler': lr_sheduler.state_dict(),
    'loss': {
        'train_loss': train_loss,
        'val_loss': val_loss,
        'best_loss': best_loss
    },
    'metric': {
        'train_acc': train_acc,
        'val_acc': val_acc,
    },
    'lr': lr_list,
    'epoch': {
        'EPOCHS': EPOCHS,
        'save_epoch': save_epoch
    }
}

torch.save(checkpoint, 'model_state_dict_01_01_24.pt')

## Загрузка модели + шедулера + метрик + эпох + другая информация


In [32]:
load_model_state = torch.load('model_state_dict_01_01_24.pt', map_location=device)

In [34]:
print(load_model_state['info'])


class MyModel(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.layer_1 = nn.Linear(input, 128)
        self.layer_2 = nn.Linear(128, output)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.act(x)
        out = self.layer_2(x)
        return out

new_model = MyModel(64*64, 2)



In [35]:
# Создание модели

new_model_1 = MyModel(64*64, 2).to(device)

loss_model = nn.MSELoss()
new_opt = torch.optim.Adam(new_model_1.parameters(), lr=0.001)
new_lr_sheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(new_opt)

In [39]:
new_model_1.load_state_dict(load_model_state['state_model'])
new_opt.load_state_dict(load_model_state['state_opt'])
new_lr_sheduler.load_state_dict(load_model_state['state_lr_scheduler'])

In [42]:
EPOCH = load_model_state['epoch']['EPOCHS']
save_epoch = load_model_state['epoch']['save_epoch']

train_loss = load_model_state['loss']['train_loss']
train_acc = load_model_state['metric']['train_acc']
val_loss = load_model_state['loss']['val_loss']
val_acc = load_model_state['metric']['val_acc']
lr_list = load_model_state['lr']

best_loss = load_model_state['loss']['best_loss']

for epoch in range(save_epoch + 1, EPOCHS):
    # Тренировка модели
    new_model_1.train()
    running_train_loss = []
    true_answer = 0
    mean_train_loss = 0
    train_loop = tqdm(train_loader, leave=False)
    for x, targets in train_loop:
        # Данные
        # (batch_size, 1, 64, 64) -> (batch_size, 64*64)
        x = x.reshape(-1, 64*64).to(device)
        targets = targets.to(torch.float32).to(device)

        # Прямой проход + расчет ошибки модели
        pred = new_model_1(x)
        loss = loss_model(pred, targets)

        # Обратный проход
        new_opt.zero_grad()
        loss.backward()
        # Шаг оптимизации
        new_opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss) / len(running_train_loss)

        true_answer += (torch.round(pred) == targets).all(dim=1).sum().item()

        train_loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}")

    # Расчет значения метрики
    running_train_acc = true_answer / len(train_set)

    # Сохранение значения функции потерь и метрики
    train_loss.append(mean_train_loss)
    train_acc.append(running_train_acc)

    # Проверка модели (валидация)
    new_model_1.eval()
    with torch.no_grad():
        running_val_loss = []
        true_answer = 0
        for x, targets in val_loader:
            # Данные
            # (batch_size, 1, 64, 64) -> (batch_size, 64*64)
            x = x.reshape(-1, 64*64).to(device)
            targets = targets.to(device)

            # Прямой проход + расчет ошибки модели
            pred = new_model_1(x)
            loss = loss_model(pred, targets)

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss)/len(running_val_loss)

            true_answer += (torch.round(pred) == targets).all(dim=1).sum().item()

    # Расчет значения метрики
    running_val_acc = true_answer / len(val_set)

    # Сохранение значения функции потерь и метрики
    val_loss.append(mean_val_loss)
    val_acc.append(running_val_acc)

    new_lr_sheduler.step(mean_val_loss)
    lr = opt.param_groups[0]['lr']
    lr_list.append(lr)

    print(f'Epoch [{epoch+1}/{EPOCHS}]: train_loss={mean_train_loss:.4f}, train_acc={running_train_acc:.4f}, '
          f'val_loss={mean_val_loss:.4f}, val_acc={running_val_acc:.4f}, lr={lr:.4f}')

    if best_loss is None:
        best_loss = mean_val_loss

    if mean_val_loss < best_loss:
        best_loss = mean_val_loss

    checkpoint = {
        'info': str_info,
        'state_model': new_model_1.state_dict(),
        'state_opt': new_opt.state_dict(),
        'state_lr_scheduler': new_lr_sheduler.state_dict(),
        'loss': {
            'train_loss': train_loss,
            'val_loss': val_loss,
            'best_loss': best_loss
        },
        'metric': {
            'train_acc': train_acc,
            'val_acc': val_acc,
        },
        'lr': lr_list,
        'epoch': {
            'EPOCHS': EPOCHS,
            'save_epoch': epoch
        }
    }

    torch.save(checkpoint, f'model_state_dict_epoch_{epoch+1}.pt')
    print(f'На эпохе - {epoch+1}, сохранена модель со значением функции потерь на валидации - {mean_val_loss:.4f}', end='\n\n')

Epoch [16/30]: train_loss=1.7930, train_acc=0.6398, val_loss=0.1539, val_acc=0.5950, lr=0.0010
На эпохе - 16, сохранена модель со значением функции потерь на валидации - 0.1539



Epoch [17/30]: train_loss=0.1435, train_acc=0.6696, val_loss=0.1298, val_acc=0.6748, lr=0.0010
На эпохе - 17, сохранена модель со значением функции потерь на валидации - 0.1298



Epoch [18/30]: train_loss=0.1236, train_acc=0.7186, val_loss=0.0676, val_acc=0.8965, lr=0.0010
На эпохе - 18, сохранена модель со значением функции потерь на валидации - 0.0676



Epoch [19/30]: train_loss=0.1056, train_acc=0.7759, val_loss=0.0625, val_acc=0.9105, lr=0.0010
На эпохе - 19, сохранена модель со значением функции потерь на валидации - 0.0625



Epoch [20/30]: train_loss=0.0852, train_acc=0.8388, val_loss=0.0754, val_acc=0.8675, lr=0.0010
На эпохе - 20, сохранена модель со значением функции потерь на валидации - 0.0754



...

Epoch [26/30]: train_loss=0.0575, train_acc=0.9263, val_loss=0.0848, val_acc=0.8641, lr=0.0010
На эпохе - 26, сохранена модель со значением функции потерь на валидации - 0.0848



Epoch [27/30]: train_loss=0.0551, train_acc=0.9335, val_loss=0.0411, val_acc=0.9672, lr=0.0010
На эпохе - 27, сохранена модель со значением функции потерь на валидации - 0.0411



Epoch [28/30]: train_loss=0.0542, train_acc=0.9341, val_loss=0.0518, val_acc=0.9420, lr=0.0010
На эпохе - 28, сохранена модель со значением функции потерь на валидации - 0.0518



Epoch [29/30]: train_loss=0.0522, train_acc=0.9419, val_loss=0.0426, val_acc=0.9724, lr=0.0010
На эпохе - 29, сохранена модель со значением функции потерь на валидации - 0.0426



Epoch [30/30]: train_loss=0.0514, train_acc=0.9421, val_loss=0.0616, val_acc=0.9062, lr=0.0010
На эпохе - 30, сохранена модель со значением функции потерь на валидации - 0.0616

